# Analisis Data Titanic

David Nehemia Sunoto  
2902726965 / PPTI 25


## 1. Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix


In [ ]:
df = pd.read_csv('titanic.csv')

print(f"Dimensi dataset: {df.shape[0]} baris dan {df.shape[1]} kolom\n")
df.head()


## 2. EDA

Di bagian ini saya memeriksa isi data, tipe kolom, nilai kosong, dan ringkasan statistik. `Survived` adalah target yang akan diprediksi: 0 berarti tidak selamat dan 1 berarti selamat.


In [ ]:
df.info()


In [ ]:
missing_df = pd.DataFrame({
    'Jumlah Kosong': df.isnull().sum(),
    'Persentase Kosong (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Tipe Data': df.dtypes
})
print("=== Pemeriksaan Data Kosong dan Duplikat ===")
print(f"Baris duplikat: {df.duplicated().sum()}\n")
missing_df


In [ ]:
df.describe()


### 2.2 Visualisasi

Histogram, boxplot, dan scatterplot di bawah ini digunakan untuk melihat sebaran usia dan tarif, serta kaitannya dengan status keselamatan.


#### A. Histogram Usia


In [ ]:
plt.figure(figsize=(8, 5))

sns.histplot(data=df, x='Age', hue='Survived', kde=True, bins=30,
             palette=['#e74c3c', '#2ecc71'], alpha=0.5)
plt.title("Histogram Usia berdasarkan Status Keselamatan")
plt.xlabel("Usia (Tahun)")
plt.ylabel("Frekuensi")
plt.legend(title='Survived', labels=['Selamat (1)', 'Tidak Selamat (0)'])

plt.tight_layout()
plt.show()


#### B. Boxplot

Boxplot membantu membandingkan sebaran usia dan tarif, termasuk nilai yang jauh dari kebanyakan data.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

sns.boxplot(data=df, x='Pclass', y='Age', palette='Set2', ax=axes[0])
axes[0].set_title("Box Plot: Distribusi Usia (Age) per Kelas (Pclass)", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Kelas Tiket (Pclass)")
axes[0].set_ylabel("Usia (Tahun)")

sns.boxplot(data=df, x='Pclass', y='Fare', hue='Survived', palette=['#e74c3c', '#2ecc71'], ax=axes[1])
axes[1].set_title("Box Plot: Tarif (Fare) per Pclass & Survived", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Kelas Tiket (Pclass)")
axes[1].set_ylabel("Tarif Tiket (Fare)")
axes[1].legend(['Tidak Selamat (0)', 'Selamat (1)'])

sns.boxplot(data=df, x='Sex', y='Age', hue='Survived', palette=['#e74c3c', '#2ecc71'], ax=axes[2])
axes[2].set_title("Box Plot: Usia per Jenis Kelamin & Survived", fontsize=12, fontweight='bold')
axes[2].set_xlabel("Jenis Kelamin (Sex)")
axes[2].set_ylabel("Usia (Tahun)")
axes[2].legend(['Tidak Selamat (0)', 'Selamat (1)'])

plt.tight_layout()
plt.show()

print("=== Median Usia Berdasarkan Pclass dan Sex ===")
display(df.groupby(['Pclass', 'Sex'])['Age'].median().unstack())


#### C. Scatterplot

Scatterplot membandingkan usia dan tarif setiap penumpang. Warna menunjukkan apakah penumpang selamat.


In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

sns.scatterplot(
    data=df,
    x='Age',
    y='Fare',
    hue='Survived',
    style='Sex',
    palette=['#e74c3c', '#2ecc71'],
    alpha=0.75,
    s=70,
    ax=ax
)

ax.set_title("Scatter Plot: Usia vs Tarif Tiket (Age vs Fare) Berdasarkan Survived & Sex", fontsize=13, fontweight='bold')
ax.set_xlabel("Usia (Age)", fontsize=11)
ax.set_ylabel("Tarif Tiket (Fare)", fontsize=11)
ax.legend(title='Survived', loc='upper right')

plt.tight_layout()
plt.show()


### 2.3 Temuan dari EDA

Pada grafik, tingkat keselamatan tampak berbeda antara perempuan dan laki-laki, serta antara kelas tiket. Usia dan tarif juga diperiksa. Ini merupakan pola pada data, bukan bukti bahwa fitur tersebut menjadi penyebab keselamatan.


---
## 3. Preprocessing

Sebelum membuat model, nilai kosong ditangani, kategori diubah menjadi angka, dan fitur numerik diskalakan.


### 3.1 Data Cleaning

Nilai `Age` yang kosong diisi dengan median, sedangkan `Embarked` diisi dengan kategori yang paling sering muncul. `Cabin` dihapus karena banyak nilainya kosong. Kolom identitas yang tidak digunakan untuk model juga dihapus. `FamilySize` dibuat dari jumlah keluarga yang ikut di kapal.


In [ ]:
df_clean = df.copy()

df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())
df_clean['Embarked'] = df_clean['Embarked'].fillna(df_clean['Embarked'].mode()[0])
df_clean['FamilySize'] = df_clean['SibSp'] + df_clean['Parch'] + 1

columns_to_drop = ['PassengerId', 'Ticket', 'Cabin', 'Name']
df_clean.drop(columns=columns_to_drop, inplace=True)

df_clean.isnull().sum()


### 3.2 Encoding

`Sex` diubah menjadi 0 atau 1. `Embarked` diubah menjadi beberapa kolom angka karena tiap pelabuhan tidak memiliki urutan tingkatan.


In [ ]:
df_encoded = df_clean.copy()
df_encoded['Sex'] = df_encoded['Sex'].map({'male': 0, 'female': 1})
df_encoded = pd.get_dummies(df_encoded, columns=['Embarked'], drop_first=True, dtype=int)
df_encoded.head()


### 3.3 Scaling Fitur Numerik

Umur, tarif, dan ukuran keluarga memiliki rentang angka berbeda. StandardScaler mengubah skala fitur tersebut agar lebih sebanding untuk Logistic Regression.


In [ ]:
num_features = ['Age', 'Fare', 'FamilySize']

scaled_preview = df_encoded.copy()
scaled_preview[num_features] = StandardScaler().fit_transform(scaled_preview[num_features])
scaled_preview[num_features].hist(figsize=(8, 5))
plt.tight_layout()
plt.show()


## 4. Modelling

Data dibagi menjadi 70% data latih dan 30% data uji. Setelah itu, Logistic Regression digunakan untuk memprediksi status keselamatan.


In [ ]:
X = df_encoded.drop(columns=['Survived']).copy()
y = df_encoded['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
scaler_model = StandardScaler()
X_train_scaled[num_features] = scaler_model.fit_transform(X_train[num_features])
X_test_scaled[num_features] = scaler_model.transform(X_test[num_features])

### 4.1 Melatih Logistic Regression


In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)
y_prob = model.predict_proba(X_test_scaled)[:, 1]

print(f"Akurasi  : {accuracy_score(y_test, y_pred):.3f}")
print(f"Presisi  : {precision_score(y_test, y_pred):.3f}")
print(f"Recall   : {recall_score(y_test, y_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix (baris = aktual, kolom = prediksi):")
print(cm)
print("TN = tidak selamat diprediksi tidak selamat")
print("FP = tidak selamat diprediksi selamat")
print("FN = selamat diprediksi tidak selamat")
print("TP = selamat diprediksi selamat")

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tidak selamat', 'Selamat'],
            yticklabels=['Tidak selamat', 'Selamat'])
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.title('Confusion Matrix')
plt.show()
